In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import wandb
from typing import List, Optional, Union, Any, Dict, Tuple
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
from tqdm import tqdm

from multilabel_classifier_evaluation import MultiLabelEval, print_dict_structure

In [ ]:
wandb.login(key=os.getenv("WANDB_API_KEY"))

In [ ]:
# Versioning stagegy used
# The last digit of model version changes if there was difference in hyperparamemer
# The middle model version number chsnges if we are using different concept of traning
# The first digit changes if the model itself changes

model_map = {
    # "alpha-1.0.1" : {
    #     "wandb_model_path": "nasa-impact/science-keywords-recommender/model-0k6954bn:final_model",
    #     "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v22",
    #     "wandb_alias": "devoted-aardvark-104",
    #     "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",
    #     "desc": "Uses Cross Entropy as Loss"
    # },
    # "alpha-1.1.1": {
    #     "wandb_model_path": "nasa-impact/science-keywords-recommender/model-6tle2hc7:final_model",
    #     "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v21",
    #     "wandb_alias": "magic-armadillo-103",
    #     "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",
    #     "desc": "Uses focal loss with gamma 4"
    # },
    # "alpha-1.1.2": {
    #     "wandb_model_path": "nasa-impact/science-keywords-recommender/model-sparb843:final_model",
    #     "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v20",
    #     "wandb_alias": "likely-eon-102",
    #     "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",
    #     "desc": "Uses focal loss with gamma 2"
    # },
    "alpha-1.2.1": {
        "wandb_model_path": "nasa-impact/science-keywords-recommender/model-odqd74ey:final_model",
        "wandb_train_data_path": "nasa-impact/science-keywords-recommender/df_train:v31",
        "wandb_val_data_path": "nasa-impact/science-keywords-recommender/df_val:v31",
        "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v31",
        "wandb_alias": "misty-valley-101",
        "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",
        "desc": "Uses focal loss with gamma 2 and uses provider-id as also the stratification strategy",
        "name": "nasa-smd-ibm-v0.1"
    },
    # "alpha-1.3.0": {
    #     "wandb_model_path": "nasa-impact/science-keywords-recommender/model-ycgwng4j:final_model",
    #     "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v24",
    #     "wandb_alias": "",
    #     "hf_pretrained_model": "FacebookAI/roberta-base",
    #     "desc": "Fine tujed roberta-base model, uses focal loss with gamma 2 and uses provider-id as also the stratification strategy"
    # },

    "alpha-1.4.1": {
        "wandb_model_path": "nasa-impact/science-keywords-recommender/model-mwexjd84:v1",
        "wandb_train_data_path": "nasa-impact/science-keywords-recommender/df_train:v31",
        "wandb_val_data_path": "nasa-impact/science-keywords-recommender/df_val:v31",
        "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v31",
        "wandb_alias": "mild-flower-149",
        "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",
        "desc": "Original Indus model: Base Frozen :has heavery regularization, warmup and cosine lr schedular",
        "name": "nasa-smd-ibm-v0.1_frozen_base"
    },
    "alpha-1.4.2": {
        "wandb_model_path": "nasa-impact/science-keywords-recommender/model-ub05r475:v1",
        "wandb_train_data_path": "nasa-impact/science-keywords-recommender/df_train:v51",
        "wandb_val_data_path": "nasa-impact/science-keywords-recommender/df_val:v51",
        "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v51",
        "test_data_frac": 0.15,
        "wandb_alias": "rose-waterfall-161",
        "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",
        "desc": "Original Indus model: Base Frozen: train-val-test_10-10-80 :has heavery regularization, warmup and cosine lr schedular",
        "name": "nasa-smd-ibm-v0.1_frozen_base_10%_train_data"
    },

    # "beta-1.4.0": {
    #     "wandb_model_path": "nasa-impact/science-keywords-recommender/model-70xr1m8r:final_model",
    #     "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v26",
    #     "wandb_alias": "",
    #     "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",
    #     "desc": "Fine tuned indus model on limited keyword scope of existing GKR model, uses focal loss with gamma 2 and uses provider-id as also the stratification strategy"
    # },
    # "gamma-1.5.1": {
    #     "wandb_model_path": "nasa-impact/science-keywords-recommender/model-djnkvj61:v1",
    #     "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v29",
    #     "wandb_alias": "gamma-1.5.1",
    #     "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",
    #     "desc": "lowe lr, Fine tuned indus model on old data, uses focal loss with gamma 2 and uses provider-id as also the stratification strategy"
    # }
    # "delta-1.6.1": {
    #     "wandb_model_path": "nasa-impact/science-keywords-recommender/model-fuvd8vec:v1",
    #     "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v31",
    #     "wandb_alias": "still-river-120",
    #     "hf_pretrained_model": "answerdotai/ModernBERT-base",  # used to load tokenizer
    #     "desc": "Uses intermediate fine tuned version of modernbert on 500k sde data: Uses focal loss with gamma 2 and uses provider-id as also the stratification strategy"
    # },
    "delta-1.6.2": {
        "wandb_model_path": "nasa-impact/science-keywords-recommender/model-fcv995sl:v1",
        "wandb_train_data_path": "nasa-impact/science-keywords-recommender/df_train:v31",
        "wandb_val_data_path": "nasa-impact/science-keywords-recommender/df_val:v31",
        "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v31",
        "wandb_alias": "pleasant-brook-144",
        "hf_pretrained_model": "answerdotai/ModernBERT-base",  # used to load tokenizer
        "desc": "Fine tuned version of modernbert on 500k sde data: has heavery regularization, warmup and cosine lr schedular",
        "name": "modernbert-sde-v0.2"
    },
    "epsilon-1.7.1": {
        "wandb_model_path": "nasa-impact/science-keywords-recommender/model-x0pmufls:v1",
        "wandb_train_data_path": "nasa-impact/science-keywords-recommender/df_train:v31",
        "wandb_val_data_path": "nasa-impact/science-keywords-recommender/df_val:v31",
        "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v31",
        "wandb_alias": "snowy-cherry-146",
        "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",  # used to load tokenizer
        "desc": "Indus model with extended context size: has heavery regularization, warmup and cosine lr schedular",
        "name": "indus-sde-v0.2"
    },
    "epsilon-1.7.2": {
        "wandb_model_path": "nasa-impact/science-keywords-recommender/model-ql616s9l:v1",
        "wandb_train_data_path": "nasa-impact/science-keywords-recommender/df_train:v51",
        "wandb_val_data_path": "nasa-impact/science-keywords-recommender/df_val:v51",
        "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v51",
        "test_data_frac": 0.15,
        "wandb_alias": "sandy-shape-162",
        "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",  # used to load tokenizer
        "desc": "Indus model with extended context size: train-val-test_10-10-80 split:has heavery regularization, warmup and cosine lr schedular",
        "name": "indus-sde-v0.2_10%_train_data"
    },
    "epsilon-1.8.1": {
        "wandb_model_path": "nasa-impact/science-keywords-recommender/model-t2o5b1ve:v1",
        "wandb_train_data_path": "nasa-impact/science-keywords-recommender/df_train:v31",
        "wandb_val_data_path": "nasa-impact/science-keywords-recommender/df_val:v31",
        "wandb_test_data_path": "nasa-impact/science-keywords-recommender/df_test:v31",
        "wandb_alias": "easy-moon-150",
        "hf_pretrained_model": "nasa-impact/nasa-smd-ibm-v0.1",  # used to load tokenizer
        "desc": "Indus model with extended context size: Base Frozen :has heavery regularization, warmup and cosine lr schedular",
        "name": "indus-sde-v0.2_frozen_base_500k_sde"
    },
}

# ORIGINAL_DATA_PATH = "../data/cmr/_cmr_collections_umm_unfiltered_one_hot_v3.parquet"
ORIGINAL_DATA_PATH = "/rhome/sawale/indus_traning/science-keywords-classification/data/cmr/_cmr_collections_umm_unfiltered_one_hot_v3.parquet"
target_df = pd.read_parquet(ORIGINAL_DATA_PATH)
# drop Abstract colun
target_df = target_df.drop(columns=["Abstract", "provider-id"])
# remanem concept-id to concept_id
target_df = target_df.rename(columns={"concept-id": "concept_id"})

In [ ]:
target_df.head()

In [ ]:
def load_wandb_artifacts(model_path, model_name, dataset_paths:List, target_df=target_df):
    api = wandb.Api()
    artifact = api.artifact(model_path)
    artifact_dir = artifact.download()
    model = AutoModelForSequenceClassification.from_pretrained(
        artifact_dir
    )
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    dfs = []
    for d_path in dataset_paths:
        artifact = api.artifact(d_path)
        artifact_dir = artifact.download()
        df = pd.read_parquet(artifact_dir)
        df = df.merge(target_df, on="concept_id", how="inner")
        dfs.append(df)
    return model, tokenizer, dfs



In [ ]:

results = {}

for m_str, values in model_map.items():
    params = {}
    name = values.get("name", m_str)
    model_wb_path = values.get("wandb_model_path")
    tokenizer_path = values.get("hf_pretrained_model")
    dataset_paths = [values.get("wandb_train_data_path"), values.get("wandb_val_data_path"), values.get("wandb_test_data_path")]
    test_data_frac = values.get("test_data_frac", None)

    model, tokenizer, dfs = load_wandb_artifacts(model_wb_path, tokenizer_path, dataset_paths)

    params["model"] = model
    params["tokenizer"] = tokenizer
    params["train_data_df"] = dfs[0]
    params["val_data_df"] = dfs[1]
    params["test_data_df"] = dfs[2] if test_data_frac is None else dfs[2].tail(int(test_data_frac * len(dfs[2])))
    params["text_col"] = "text"
    params["label_cols"] = [c for c in target_df.columns if c not in ["concept_id"]]

    eval = MultiLabelEval(
        **params,
        model_name=m_str,
        thresholds=[0.3, 0.5, 0.7],
        top_ks=[1, 5, 10],
        n_rows=512 # for testing
    )
    result = eval.evaluate_in_memory()
    results[name] = result
    # print_dict_structure(result)




In [ ]:
MultiLabelEval.plot_metrics_comparision_with_splits(results, output_dir = "./metrics_comparision_with_splits")

In [ ]:

%reload_ext autoreload
%autoreload 2

from multilabel_classifier_evaluation import MultiLabelEval, print_dict_structure
import pandas as pd

metric_dfs = {
    "k": pd.read_parquet("/rhome/sawale/multilabel-eval/metrics_comparision_with_splits/k/k_all_metrics.parquet"),
    "threshold": pd.read_parquet("/rhome/sawale/multilabel-eval/metrics_comparision_with_splits/threshold/threshold_all_metrics.parquet"),
}


renaming_model = {
    'nasa-smd-ibm-v0.1': 'original-indus', 
    'nasa-smd-ibm-v0.1_frozen_base': 'original-indus (frozen)',
    'nasa-smd-ibm-v0.1_frozen_base_10%_train_data': 'original-indus (frozen + subsampled_data)',
    'modernbert-sde-v0.2': 'modernbert-sde-v0.2', 
    'indus-sde-v0.2': 'indus-sde-v0.2',
    'indus-sde-v0.2_10%_train_data': 'indus-sde-v0.2 (subsampled_data)',
    'indus-sde-v0.2_frozen_base_500k_sde': 'indus-sde-v0.2 (frozen)'
}

metric_dfs["k"]["model"] = metric_dfs["k"]["model"].replace(renaming_model)
metric_dfs["threshold"]["model"] = metric_dfs["threshold"]["model"].replace(renaming_model)

MultiLabelEval.plot_comparision(results=None, output_dir = "./test", 
                                                    project_name="GKR", ledgend_n_cols=3, metric_dfs=metric_dfs,)